In [1]:
model_path1 = f"/home/jupyter/repos/mmelon/biomed-multi-view/model/pgk2_del_cdd_10epochs/last.ckpt"
testpath= f"/home/jupyter/repos/mmelon/biomed-multi-view/notebooksdata/finetuning/PGK2_DEL_cdd_to_creative/train.csv"
outpath= f"/home/jupyter/repos/mmelon/biomed-multi-view/EmbeddingOutput/PGK2_DEL_cdd_to_creative"
out_total=f"/home/jupyter/repos/mmelon/biomed-multi-view/EmbeddingOutput/PGK2_DEL_cdd_to_creative/multiview-mmelon.parquet"

In [2]:
import torch
import torch.nn as nn
from bmfm_sm.predictive.data_modules.graph_finetune_dataset import Graph2dFinetuneDataPipeline
from bmfm_sm.predictive.data_modules.image_finetune_dataset import ImageFinetuneDataPipeline
from bmfm_sm.predictive.data_modules.text_finetune_dataset import TextFinetuneDataPipeline
from joblib import Parallel, delayed
from torch.nn.functional import pad
from torch.nn.utils.rnn import pad_sequence
from bmfm_sm.api.smmv_api import SmallMoleculeMultiViewModel, PredictionIterator
from bmfm_sm.core.data_modules.namespace import LateFusionStrategy
from bmfm_sm.api.dataset_registry import DatasetRegistry

2026-03-19 03:10:00,416 - rdkit - INFO - instance-20260302-162623:139891838498624:0:0 - Enabling RDKit 2022.09.5 jupyter extensions


In [3]:
class BiomedMultiViewEncoder(nn.Module):
    def __init__(self, model_path=None):
        super(BiomedMultiViewEncoder, self).__init__()
        # Initialize the pretrained model
        biomed_smmv_pretrained = SmallMoleculeMultiViewModel.from_pretrained(
            LateFusionStrategy.ATTENTIONAL, inference_mode=True, model_path=model_path
        )
        # Initialize the model subcomponents
        self.model_graph = biomed_smmv_pretrained.model_graph  # output dim: 512
        self.model_image = biomed_smmv_pretrained.model_image  # output dim: 512
        self.model_text = biomed_smmv_pretrained.model_text  # output dim: 768

    # Helper function for collating the individual processed graph samples:
    @staticmethod
    def collate_graph_data(graph_data_list):
        collated = {}
        collated["node_num"] = torch.cat(
            [sample["node_num"] for sample in graph_data_list]
        )
        collated["node_data"] = torch.cat(
            [sample["node_data"] for sample in graph_data_list]
        )
        collated["edge_num"] = torch.cat(
            [sample["edge_num"] for sample in graph_data_list]
        )
        collated["edge_data"] = torch.cat(
            [sample["edge_data"] for sample in graph_data_list]
        )
        collated["edge_index"] = torch.cat(
            [sample["edge_index"] for sample in graph_data_list], dim=1
        )

        max_node_num = max(collated["node_num"])
        collated["lap_eigvec"] = torch.cat(
            [
                pad(i, (0, max_node_num - i.size(1)), value=float("0"))
                for i in [sample["lap_eigvec"] for sample in graph_data_list]
            ]
        )
        return collated

    def process_smiles(sm):
        img_data = ImageFinetuneDataPipeline.smiles_to_image_format(sm)
        txt_data = TextFinetuneDataPipeline.smiles_to_text_format(sm)
        graph_data = Graph2dFinetuneDataPipeline.smiles_to_graph_format(sm)

        return {
            "img": img_data["img"].squeeze(0),
            "tokens": txt_data["smiles.tokenized"].squeeze(0),
            "attention_mask": txt_data["attention_mask"].squeeze(0),
            "graph": graph_data,
        }

    def forward(self, smiles: list, batch_process: bool = False):
        if batch_process:
            results = Parallel(n_jobs=8)(
                delayed(BiomedMultiViewEncoder.process_smiles)(sm) for sm in smiles
            )
        else:
            results = [BiomedMultiViewEncoder.process_smiles(sm) for sm in smiles]

        image_tensors = [r["img"] for r in results]
        tokenized_smiles_list = [r["tokens"] for r in results]
        attention_mask_list = [r["attention_mask"] for r in results]
        graph_data_list = [r["graph"] for r in results]

        image_batch = torch.stack(image_tensors, dim=0)
        tokenized_smiles_batch = pad_sequence(tokenized_smiles_list, batch_first=True)
        attention_mask_batch = pad_sequence(attention_mask_list, batch_first=True)
        graph_batch = BiomedMultiViewEncoder.collate_graph_data(graph_data_list)

        image_emb = self.model_image(image_batch)
        text_emb = self.model_text(tokenized_smiles_batch, attention_mask_batch)
        graph_emb = self.model_graph(graph_batch)

        return [graph_emb, image_emb, text_emb]

In [4]:
# read hugging face model
model = SmallMoleculeMultiViewModel.from_pretrained(LateFusionStrategy.ATTENTIONAL,
                                                    model_path='ibm/biomed.sm.mv-te-84m',
                                                    huggingface=True)

2026-03-19 03:10:01,035 - root - INFO - instance-20260302-162623:139891838498624:0:0 - Loading checkpoint via HuggingFace Hub from provided path ibm/biomed.sm.mv-te-84m
/opt/conda/envs/biomed-multiview/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/opt/conda/envs/biomed-multiview/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
2026-03-19 03:10:02,039 - root - INFO - instance-20260302-162623:139891838498624:0:0 - Using coeff_mlp architecture for aggregator
2026-03-19 03:10:02,040 - root - INFO - instance-20260302-162623:139891838498624:0:0 - dim_list [512, 512, 768] for aggregator
2026-03-19 03:10:02,

In [5]:
# Reading local model
model = SmallMoleculeMultiViewModel.from_pretrained(LateFusionStrategy.ATTENTIONAL,
                                                    model_path=model_path1,
                                                    huggingface=False)

2026-03-19 03:10:03,077 - root - INFO - instance-20260302-162623:139891838498624:0:0 - Using coeff_mlp architecture for aggregator
2026-03-19 03:10:03,078 - root - INFO - instance-20260302-162623:139891838498624:0:0 - dim_list [512, 512, 768] for aggregator
2026-03-19 03:10:03,097 - root - INFO - instance-20260302-162623:139891838498624:0:0 - Loading checkpoint from provided path /home/jupyter/repos/mmelon/biomed-multi-view/model/pgk2_del_cdd_10epochs/last.ckpt
2026-03-19 03:10:03,602 - root - INFO - instance-20260302-162623:139891838498624:0:0 - Loading finetune checkpoint for SmallMoleculeMultiView Model - <All keys matched successfully>
2026-03-19 03:10:03,605 - root - INFO - instance-20260302-162623:139891838498624:0:0 - in train False setting deterministic_eval = True


In [6]:
# Get 512 dimenstion embedding
example_smiles = "CC(C)CC1=CC=C(C=C1)C(C)C(=O)O"
example_emb = SmallMoleculeMultiViewModel.get_embeddings(
    smiles=example_smiles,
    model_path="ibm/biomed.sm.mv-te-84m",
    huggingface=True,
)
print(example_emb.shape)

2026-03-19 03:10:03,618 - root - INFO - instance-20260302-162623:139891838498624:0:0 - Loading checkpoint via HuggingFace Hub from provided path ibm/biomed.sm.mv-te-84m
2026-03-19 03:10:04,520 - root - INFO - instance-20260302-162623:139891838498624:0:0 - Using coeff_mlp architecture for aggregator
2026-03-19 03:10:04,521 - root - INFO - instance-20260302-162623:139891838498624:0:0 - dim_list [512, 512, 768] for aggregator
2026-03-19 03:10:04,676 - root - INFO - instance-20260302-162623:139891838498624:0:0 - in train False setting deterministic_eval = True


torch.Size([512])


In [7]:
example_smiles = "CC(C)CC1=CC=C(C=C1)C(C)C(=O)O"
example_emb = SmallMoleculeMultiViewModel.get_embeddings(
    smiles=example_smiles,
    model_path= model_path1,
    huggingface=False,
)
print(example_emb.shape)

2026-03-19 03:10:05,988 - root - INFO - instance-20260302-162623:139891838498624:0:0 - Using coeff_mlp architecture for aggregator
2026-03-19 03:10:05,989 - root - INFO - instance-20260302-162623:139891838498624:0:0 - dim_list [512, 512, 768] for aggregator
2026-03-19 03:10:06,006 - root - INFO - instance-20260302-162623:139891838498624:0:0 - Loading checkpoint from provided path /home/jupyter/repos/mmelon/biomed-multi-view/model/pgk2_del_cdd_10epochs/last.ckpt
2026-03-19 03:10:06,334 - root - INFO - instance-20260302-162623:139891838498624:0:0 - Loading finetune checkpoint for SmallMoleculeMultiView Model - <All keys matched successfully>
2026-03-19 03:10:06,336 - root - INFO - instance-20260302-162623:139891838498624:0:0 - in train False setting deterministic_eval = True


torch.Size([512])


In [8]:
model = BiomedMultiViewEncoder(model_path = model_path1)

2026-03-19 03:10:07,302 - root - INFO - instance-20260302-162623:139891838498624:0:0 - Using coeff_mlp architecture for aggregator
2026-03-19 03:10:07,303 - root - INFO - instance-20260302-162623:139891838498624:0:0 - dim_list [512, 512, 768] for aggregator
2026-03-19 03:10:07,319 - root - INFO - instance-20260302-162623:139891838498624:0:0 - Loading checkpoint from provided path /home/jupyter/repos/mmelon/biomed-multi-view/model/pgk2_del_cdd_10epochs/last.ckpt
2026-03-19 03:10:07,646 - root - INFO - instance-20260302-162623:139891838498624:0:0 - Loading finetune checkpoint for SmallMoleculeMultiView Model - <All keys matched successfully>
2026-03-19 03:10:07,648 - root - INFO - instance-20260302-162623:139891838498624:0:0 - in train False setting deterministic_eval = True


In [9]:
import pandas as pd
smiles_list = list(pd.read_csv(testpath)['smiles'])
#print(smils_list[1:5])

In [10]:
# Getting embeddings without batch processing (Faster for smaller lists due to lack of parallelization set-up)
# single_view_embeddings2 = model.forward(smiles_list)

In [11]:
# Getting embeddings with batch processing (Faster for larger lists)
single_view_embeddings = model.forward(smiles_list, batch_process=True)

RuntimeError: [enforce fail at alloc_cpu.cpp:83] err == 0. DefaultCPUAllocator: can't allocate memory: you tried to allocate 79032418304 bytes. Error code 12 (Cannot allocate memory)

In [ ]:
# Graph Embeddings, Image Embeddings, Text Embeddings
single_view_embeddings[0].shape, single_view_embeddings[1].shape, single_view_embeddings[2].shape
print(single_view_embeddings[0].shape)

In [ ]:
import pandas as pd
import os
import numpy as np

# Assuming you already have smiles_list from before
# smiles_list = [...]

output_dir = outpath
os.makedirs(output_dir, exist_ok=True)

names_and_tensors = [
    ("graph-mmelon.parquet", single_view_embeddings[0]),
    ("image-mmelon.parquet", single_view_embeddings[1]),
    ("text-mmelon.parquet", single_view_embeddings[2]),
]

for filename, tensor in names_and_tensors:
    arr = tensor.detach().cpu().numpy()  # shape: (N, D)

    # Convert each row to a list (important!)
    embeddings_list = [row.tolist() for row in arr]

    df = pd.DataFrame({
        "smiles": smiles_list,
        "embedding": embeddings_list
    })

    df.to_parquet(os.path.join(output_dir, filename), index=False)

print("Saved all three parquet files with embeddings as arrays.")

In [ ]:
import os
import numpy as np
import pandas as pd
from tqdm import tqdm

# Read SMILES
smiles_list = list(
    pd.read_csv(testpath)["smiles"]
)

# Output path
output_path = out_total

# Collect embeddings
all_embeddings = []

for smi in tqdm(smiles_list):
    emb = SmallMoleculeMultiViewModel.get_embeddings(
        smiles=smi,
        model_path=model_path1,
        huggingface=False,
    )

    if hasattr(emb, "detach"):
        emb = emb.detach().cpu().numpy()

    emb = np.asarray(emb).reshape(-1)

    # convert to list here ✅
    all_embeddings.append(emb.tolist())

# Create dataframe (NO vstack!)
df = pd.DataFrame({
    "smiles": smiles_list,
    "embedding": all_embeddings
})

df.to_parquet(output_path, index=False)

print("Saved to:", output_path)
print("Shape:", df.shape)

In [ ]:
model_path1 = f"/home/jupyter/repos/mmelon/biomed-multi-view/model/pgk2_del_cdd_10epochs/last.ckpt"
testpath= f"/home/jupyter/repos/mmelon/biomed-multi-view/notebooksdata/inference/temp.csv"
outpath= f"/home/jupyter/repos/mmelon/biomed-multi-view/EmbeddingOutput/temp"
out_total=f"/home/jupyter/repos/mmelon/biomed-multi-view/EmbeddingOutput/temp/multiview-mmelon.parquet"